# 08 · Recursion and Backtracking

After this notebook you can write a recursive function with a correct base case, picture the call stack it builds, make it fast with memoisation, and solve the classic "generate all ..." problems (subsets, permutations, combination sum, parentheses, N-Queens, word search) with one backtracking template.

**Time:** ~35 min · **Runs:** offline on CPU in < 1 min · **Needs:** [Big-O and timing](01_big_o_and_timing.ipynb), [Stacks and queues](04_stacks_and_queues.ipynb)

## Why this matters in interviews

- "Return **all** subsets / orderings / combinations / valid strings / placements" is one family. They are all the same backtracking template with a different "what can I choose next?" rule; spotting that is most of the battle.
- Recursion underpins what comes next: tree problems ([09](09_trees_and_bst.ipynb)), graph DFS ([11](11_graphs_bfs_dfs_topo_dijkstra.ipynb)), and dynamic programming ([12](12_dynamic_programming.ipynb)). Memoised recursion *is* top-down DP.
- The follow-ups are predictable: What is the base case? How deep does the stack get? What happens at n = 10,000 in Python? How do you prune? And the space complexity nobody mentions: the call stack.

## What interviewers ask

- "Write Fibonacci / factorial recursively. What is its complexity? Make it fast." (memoisation)
- "Return all subsets of a set" (LeetCode 78), "all permutations" (46), "all combinations that sum to a target" (39).
- "Generate all valid combinations of n pairs of parentheses" (22).
- "Place n queens on an n×n board so that none attack each other" (51 / 52).
- "Does this word exist in the letter grid?" (79)
- Follow-ups: "What is the space complexity?", "What breaks for very deep input?", "Where can you prune?", "Can you do it iteratively?"

In [ ]:
import sys
import itertools
import math
import time
from collections import Counter
from functools import lru_cache

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e5e0", "axes.axisbelow": True, "axes.titlesize": 11,
                     "legend.frameon": False})
BLUE, ORANGE, AQUA, MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#52514e"
print("Python", sys.version.split()[0])

## 1. Recursion: a function that calls a smaller copy of itself

**In plain English:** to solve a big problem, solve a slightly smaller version of the same problem and use its answer. Keep shrinking until the problem is so small that the answer is obvious. That tiny case is the **base case**; the shrink-and-combine step is the **recursive case**.

Two questions decide whether a recursive function is correct:

1. **Base case:** does the smallest input (0, 1, empty list, empty tree) return the right answer *without* recursing?
2. **Progress:** does every recursive call move strictly closer to a base case? If not, it never stops.

Let us trace `factorial(4)` and record every call and return.

In [ ]:
trace = []   # (event, depth, n, value)

def factorial(n, depth=0):
    trace.append(("call", depth, n, None))
    if n <= 1:                                   # base case: answer known directly
        result = 1
    else:                                        # recursive case: shrink, then combine
        result = n * factorial(n - 1, depth + 1)
    trace.append(("return", depth, n, result))
    return result

assert factorial(4) == 24
for event, depth, n, value in trace:
    text = f"call factorial({n})" if event == "call" else f"factorial({n}) returns {value}"
    print("    " * depth + text)
max_depth = max(d for _, d, _, _ in trace) + 1
print(f"\nframes on the stack at the deepest point: {max_depth}")
assert max_depth == 4

### The call stack, as a picture

Every call gets a **frame**: its own copy of the local variables (`n`) and a bookmark of where to resume. A frame cannot finish until the call above it returns. So at the deepest moment all four frames are alive at once, and each is waiting. Then the returns flow back down: 1 → 2 → 6 → 24.

That stack is why recursion costs **O(depth) extra memory** even when you allocate nothing yourself.

In [ ]:
def draw_stack(trace):
    calls = [n for ev, _, n, _ in trace if ev == "call"]           # bottom -> top
    returns = {n: v for ev, _, n, v in trace if ev == "return"}
    fig, ax = plt.subplots(figsize=(9, 3.6))
    for i, n in enumerate(calls):
        top = i == len(calls) - 1
        ax.add_patch(FancyBboxPatch((0, i), 5.6, 0.8, boxstyle="round,pad=0.04",
                                    fc="#fde7dc" if top else "#e7f0ff", ec=ORANGE if top else BLUE))
        label = f"factorial({n})   n={n}   " + ("base case" if top else f"waiting for factorial({n - 1})")
        ax.text(0.15, i + 0.4, label, va="center", fontsize=9.5, family="monospace")
        ax.text(5.9, i + 0.4, f"returns {returns[n]}", va="center", fontsize=9.5, color=MUTED)
    ax.annotate("", xy=(7.6, 0.3), xytext=(7.6, len(calls) - 0.5),
                arrowprops=dict(arrowstyle="->", lw=1.6, color=MUTED))
    ax.text(7.75, len(calls) / 2 - 0.2, "values flow\nback down", fontsize=9, color=MUTED, va="center")
    ax.set_xlim(-0.2, 9); ax.set_ylim(-0.3, len(calls) + 0.1); ax.axis("off")
    ax.set_title("Call stack at its deepest point: the newest frame (top) is the base case")
    plt.show()

draw_stack(trace)

### The recursion limit

Python does **not** optimise tail calls, and it caps the stack depth (default 1000 frames) so a runaway recursion raises `RecursionError` instead of crashing the process. The notebook machinery already sits on a couple of dozen frames, so you get slightly fewer than the limit.

In [ ]:
print("sys.getrecursionlimit():", sys.getrecursionlimit())

def deepest_possible():
    depth = 0
    def dive():
        nonlocal depth
        depth += 1
        dive()
    try:
        dive()
    except RecursionError:
        return depth

print("frames we could stack before RecursionError:", deepest_possible())

def sum_to_recursive(n):
    return 0 if n == 0 else n + sum_to_recursive(n - 1)

def sum_to_iterative(n):
    total = 0
    for i in range(1, n + 1):
        total += i
    return total

try:
    sum_to_recursive(10_000)
    raise AssertionError("expected RecursionError")
except RecursionError:
    print("sum_to_recursive(10_000) -> RecursionError")
assert sum_to_iterative(10_000) == 10_000 * 10_001 // 2
print("sum_to_iterative(10_000) ->", sum_to_iterative(10_000))

**Fixes, in order of preference:** (1) rewrite with a loop or an explicit stack (the stack you manage is a Python list, limited only by memory); (2) restructure so the depth is `O(log n)` (divide in halves); (3) `sys.setrecursionlimit(...)` only as a last resort — recursion that passes through C code (an `lru_cache` wrapper, `sorted(key=...)`, `__repr__`) still uses the real C stack, and overflowing that crashes the interpreter instead of raising an error.

### Recursion on nested data

Nested dicts and lists (JSON, configs) are recursive structures, so the natural code that walks them is recursive. Two classic exercises: flatten the keys, and measure the nesting depth. Even the C JSON parser has a nesting limit: a hostile `[[[[...]]]]` payload raises `RecursionError` (in CPython 3.12 at a few thousand levels, and `sys.setrecursionlimit` does not change it).

In [ ]:
def max_depth(obj):
    """Nesting depth of a JSON-like value: scalars are 0, every dict/list level adds 1."""
    if isinstance(obj, dict):
        return 1 + max((max_depth(v) for v in obj.values()), default=0)
    if isinstance(obj, list):
        return 1 + max((max_depth(v) for v in obj), default=0)
    return 0                                           # base case: a scalar

def flatten(obj, prefix=""):
    """{'a': {'b': 1}} -> {'a.b': 1}; list items use their index as the key."""
    if not isinstance(obj, (dict, list)):
        return {prefix: obj}                           # base case: a leaf value
    items = obj.items() if isinstance(obj, dict) else enumerate(obj)
    out = {}
    for key, value in items:
        out.update(flatten(value, f"{prefix}.{key}" if prefix else str(key)))
    return out

trip = {"city": "Paris", "dates": {"from": "2026-10-01", "to": "2026-10-03"},
             "filters": [{"type": "hotel", "stars": 4}]}
print("flattened:", flatten(trip))
print("max depth:", max_depth(trip))
assert flatten(trip)["filters.0.stars"] == 4 and max_depth(trip) == 3
assert max_depth({}) == 1 and max_depth(7) == 0 and flatten({}) == {}

import json
try:
    json.loads("[" * 100_000 + "]" * 100_000)
except RecursionError as e:
    print("hostile payload ->", type(e).__name__)

## 2. Memoisation: remember answers you already computed

**In plain English:** naive recursive Fibonacci asks for `fib(3)` many times, and every time it recomputes it from scratch. Memoisation keeps a dictionary of "input → answer"; the second time you ask, it is a lookup.

`functools.lru_cache(maxsize=None)` (or its alias `functools.cache`) does this for you. Let us count calls for both versions.

In [ ]:
calls = 0
def fib_naive(n):
    global calls
    calls += 1
    return n if n < 2 else fib_naive(n - 1) + fib_naive(n - 2)

@lru_cache(maxsize=None)
def fib_memo(n):
    return n if n < 2 else fib_memo(n - 1) + fib_memo(n - 2)

ns = list(range(2, 26))
naive_calls, memo_calls = {}, {}
for n in ns:
    calls = 0
    fib_naive(n)
    naive_calls[n] = calls
    fib_memo.cache_clear()
    fib_memo(n)
    info = fib_memo.cache_info()
    memo_calls[n] = info.hits + info.misses
    assert info.misses == n + 1                         # every fib(k), k = 0..n, computed exactly once
    assert memo_calls[n] == 2 * n - 1

for n in (10, 20, 25):
    assert naive_calls[n] == 2 * fib_memo(n + 1) - 1    # the call count is itself Fibonacci-shaped
    print(f"fib({n}) = {fib_memo(n):>6}   naive calls: {naive_calls[n]:>7,}   memo calls: {memo_calls[n]}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ns, [naive_calls[n] for n in ns], color=BLUE, lw=2, marker="o", ms=4)
ax.plot(ns, [memo_calls[n] for n in ns], color=ORANGE, lw=2, marker="o", ms=4)
ax.set_yscale("log")
ax.text(ns[-1] + 0.3, naive_calls[ns[-1]], "naive\n~1.6^n", color=BLUE, va="center")
ax.text(ns[-1] + 0.3, memo_calls[ns[-1]], "memoised\n2n - 1", color=ORANGE, va="center")
ax.set_xlim(0, ns[-1] + 3)
ax.set_xlabel("n"); ax.set_ylabel("function calls (log scale)")
ax.set_title("Computing fib(n): exponential calls without a cache, linear with one")
plt.show()

Read the chart: on a log scale an exponential is a straight line. Naive Fibonacci grows about 1.6× per step (the golden ratio), so `fib(25)` needs a quarter of a million calls; the memoised version needs 49.

**Gotchas interviewers probe:**

| Gotcha | What happens | Fix |
|---|---|---|
| Unhashable arguments | `lru_cache` raises `TypeError` for a list argument | pass a tuple, or memoise on an index |
| The cache lives forever | memory grows; stale results across test cases | `fn.cache_clear()`, or a bounded `maxsize` |
| Top-down memo is still deep | `fib_memo(5000)` from cold still recurses 5000 frames | warm up bottom-up, or write the loop (bottom-up DP, [notebook 12](12_dynamic_programming.ipynb)) |
| `def f(x, memo={})` | the default dict is shared across calls (sometimes on purpose, usually a bug) | `memo=None`, then create it inside |

In [ ]:
fib_memo.cache_clear()
try:
    fib_memo(5000)
    raise AssertionError("expected RecursionError")
except RecursionError:
    print("fib_memo(5000) from a cold cache -> RecursionError")

for k in range(5001):          # warm the cache bottom-up: each call only recurses one level
    fib_memo(k)
print("after warming bottom-up, fib(5000) has", len(str(fib_memo(5000))), "digits")

## 3. Backtracking: the template

**In plain English:** backtracking walks a tree of decisions depth-first. At each step you **choose** an option, **explore** everything that follows from it (recursion), then **un-choose** it so the next option starts from a clean slate. If a partial answer can already never become valid, stop exploring that branch (**prune**).

When to recognise it: the problem says "all", "every", "list the combinations", "can you place", "does a path exist", and the input is small (n ≤ ~20, a board ≤ ~10×10). The output itself is exponential, so no algorithm can be polynomial.

```python
def solve(input):
    results, path = [], []
    def backtrack(state):
        if is_complete(path):
            results.append(path[:])      # COPY: path keeps changing after this line
            return
        for choice in choices(state):
            if not is_valid(path, choice):
                continue                 # prune
            path.append(choice)          # choose
            backtrack(next_state)        # explore
            path.pop()                   # un-choose
    backtrack(start_state)
    return results
```

The decision tree for subsets of `[1, 2, 3]` (every node is an answer; children only take elements to the right, so `[2, 1]` is never produced as a duplicate of `[1, 2]`):

```
[]
├── [1]
│   ├── [1, 2]
│   │   └── [1, 2, 3]
│   └── [1, 3]
├── [2]
│   └── [2, 3]
└── [3]
```

**Complexity rule of thumb:** time = (number of nodes in the tree) × (work per node). Space = recursion depth + the current path (the output is extra).

### Problem 1 — Subsets / power set (LeetCode 78)

Given distinct integers, return every subset. `[1, 2, 3]` → 8 subsets. Brute force: every subset is a bitmask from `0` to `2^n - 1`; read the bits. Backtracking produces the same set and generalises to the harder variants.

In [ ]:
def subsets(nums):
    result, path = [], []
    def backtrack(start):
        result.append(path[:])              # every node of the tree is a subset
        for i in range(start, len(nums)):
            path.append(nums[i])            # choose
            backtrack(i + 1)                # explore: only elements to the right
            path.pop()                      # un-choose
    backtrack(0)
    return result

def subsets_bitmask(nums):
    n = len(nums)
    return [[nums[i] for i in range(n) if mask >> i & 1] for mask in range(1 << n)]

print(subsets([1, 2, 3]))
for nums in ([], [5], [1, 2, 3], list(range(10))):
    got = subsets(nums)
    assert len(got) == 2 ** len(nums)
    assert sorted(map(tuple, got)) == sorted(map(tuple, subsets_bitmask(nums)))
print("subsets == bitmask brute force; count is 2^n for n = 0, 1, 3, 10")

**Complexity:** O(n · 2^n) time (2^n subsets, each copied in O(n)), O(n) extra space for the recursion and the path.

**The #1 backtracking bug:** appending `path` instead of `path[:]`. You store eight references to the *same* list, which is empty by the time you look.

In [ ]:
def subsets_buggy(nums):
    result, path = [], []
    def backtrack(start):
        result.append(path)                 # BUG: the same list object every time
        for i in range(start, len(nums)):
            path.append(nums[i]); backtrack(i + 1); path.pop()
    backtrack(0)
    return result

buggy = subsets_buggy([1, 2, 3])
print("buggy result:", buggy)
assert len(buggy) == 8 and all(s == [] for s in buggy)

### Problem 2 — Permutations (LeetCode 46)

All orderings of distinct integers: `[1, 2, 3]` → 6. Now every level may pick **any unused** element, so we track `used`. Brute force would be `itertools.permutations` (fine in real code, but the interviewer wants to see the recursion); we use it as the reference answer.

In [ ]:
def permutations(nums):
    result, path, used = [], [], [False] * len(nums)
    def backtrack():
        if len(path) == len(nums):
            result.append(path[:])
            return
        for i, x in enumerate(nums):
            if used[i]:
                continue
            used[i] = True; path.append(x)      # choose
            backtrack()                         # explore
            path.pop(); used[i] = False         # un-choose
    backtrack()
    return result

print(permutations([1, 2, 3]))
for nums in ([], [7], [1, 2, 3], [4, 1, 3, 2, 5, 0]):
    got = permutations(nums)
    assert len(got) == math.factorial(len(nums))
    assert sorted(map(tuple, got)) == sorted(itertools.permutations(nums))
print("matches itertools.permutations; count is n!")

**Complexity:** O(n · n!) time, O(n) extra space. With duplicates in the input (LeetCode 47), sort first and skip a value equal to its left neighbour when that neighbour is unused — that is exercise 1 below.

### Problem 3 — Combination sum (LeetCode 39)

Distinct positive `candidates`, each usable any number of times; return every combination that sums to `target`. `candidates = [2, 3, 6, 7], target = 7` → `[[2, 2, 3], [7]]`.

Two details make it work: recurse with `i` (not `i + 1`) so the same number can be reused, and a `start` index so `[2, 2, 3]` and `[3, 2, 2]` are not both produced. **Pruning:** sort the candidates; the moment one is bigger than what remains, every later one is too, so `break`. The version without pruning recurses and only notices the overshoot one level later. Let us count visited nodes.

In [ ]:
def combination_sum(candidates, target, prune=True):
    candidates = sorted(candidates)
    result, path, visited = [], [], 0
    def backtrack(start, remaining):
        nonlocal visited
        visited += 1
        if remaining < 0:                   # only reachable without pruning
            return
        if remaining == 0:
            result.append(path[:])
            return
        for i in range(start, len(candidates)):
            if prune and candidates[i] > remaining:
                break                       # sorted, so every later candidate overshoots too
            path.append(candidates[i])
            backtrack(i, remaining - candidates[i])   # i, not i + 1: reuse allowed
            path.pop()
    backtrack(0, target)
    return result, visited

print(combination_sum([2, 3, 6, 7], 7)[0])
assert combination_sum([2, 3, 6, 7], 7)[0] == [[2, 2, 3], [7]]
assert combination_sum([2], 1)[0] == [] and combination_sum([1], 2)[0] == [[1, 1]]

res_p, nodes_p = combination_sum([2, 3, 5, 7, 11], 30, prune=True)
res_u, nodes_u = combination_sum([2, 3, 5, 7, 11], 30, prune=False)
assert res_p == res_u and all(sum(c) == 30 for c in res_p)
assert len({tuple(c) for c in res_p}) == len(res_p)            # no duplicates
assert nodes_p < nodes_u / 2                                   # pruning more than halves the work here
print(f"target 30: {len(res_p)} combinations; nodes visited with pruning {nodes_p:,} vs without {nodes_u:,}")

**Complexity:** exponential — roughly O(n^(T/m)) where T is the target and m the smallest candidate (the tree depth is at most T/m). Space O(T/m). Pruning does not change the worst case, but here it cuts the visited nodes by more than half.

### Problem 4 — Generate parentheses (LeetCode 22)

All valid strings of `n` pairs. `n = 3` → `((()))`, `(()())`, `(())()`, `()(())`, `()()()`.

**Brute force:** generate all 2^(2n) strings of `(` and `)` and keep the balanced ones. **Backtracking:** build left to right and never create an invalid prefix: add `(` while fewer than `n` are open, add `)` only while it closes something. Every leaf is then valid, so no work is wasted on dead ends. (Strings are immutable, so passing `s + "("` makes the "un-choose" automatic.)

In [ ]:
def generate_parentheses(n):
    result, nodes = [], 0
    def backtrack(s, opened, closed):
        nonlocal nodes
        nodes += 1
        if len(s) == 2 * n:
            result.append(s)
            return
        if opened < n:
            backtrack(s + "(", opened + 1, closed)
        if closed < opened:
            backtrack(s + ")", opened, closed + 1)
    backtrack("", 0, 0)
    return result, nodes

def is_balanced(s):
    bal = 0
    for ch in s:
        bal += 1 if ch == "(" else -1
        if bal < 0:
            return False
    return bal == 0

def parentheses_brute(n):
    return [s for s in map("".join, itertools.product("()", repeat=2 * n)) if is_balanced(s)]

print(generate_parentheses(3)[0])
assert generate_parentheses(3)[0] == ["((()))", "(()())", "(())()", "()(())", "()()()"]
catalan = lambda n: math.comb(2 * n, n) // (n + 1)
for n in range(1, 11):
    got, _ = generate_parentheses(n)
    assert len(got) == catalan(n) and all(map(is_balanced, got))
    if n <= 6:
        assert sorted(got) == sorted(parentheses_brute(n))
print("counts are the Catalan numbers:", [catalan(n) for n in range(1, 11)])

In [ ]:
n_values = list(range(1, 11))
brute = [4 ** n for n in n_values]                        # 2^(2n) candidate strings
bt_nodes = [generate_parentheses(n)[1] for n in n_values]
valid = [catalan(n) for n in n_values]

fig, ax = plt.subplots(figsize=(8, 4))
for ys, color, label in [(brute, BLUE, "brute force: strings generated (4^n)"),
                         (bt_nodes, ORANGE, "backtracking: nodes visited"),
                         (valid, AQUA, "valid answers (Catalan)")]:
    ax.plot(n_values, ys, color=color, lw=2, marker="o", ms=4, label=label)
ax.set_yscale("log"); ax.legend()
ax.set_xlabel("n (pairs of parentheses)"); ax.set_ylabel("count (log scale)")
ax.set_title("Pruning invalid prefixes: backtracking work tracks the answer count, not 4^n")
plt.show()
print(f"n = 10: brute force {brute[-1]:,} strings vs backtracking {bt_nodes[-1]:,} nodes for {valid[-1]:,} answers")

**Complexity:** the number of answers is the n-th Catalan number ≈ 4^n / (n^1.5 · √π); each costs O(n) to build, so O(4^n / √n) time and O(n) depth.

### Problem 5 — N-Queens (LeetCode 51)

Place `n` queens on an n×n board so no two share a row, column or diagonal. Go **row by row** (so rows never clash) and keep three sets of attacked lines: columns `c`, diagonals `r - c` and anti-diagonals `r + c` — each is constant along its line, so checking a square is O(1).

**Brute force:** one queen per row and per column means each placement is a permutation of columns: check all n! of them. Backtracking abandons a partial board as soon as two queens attack.

In [ ]:
def solve_n_queens(n):
    solutions, queens, nodes = [], [], 0          # queens[r] = column of the queen in row r
    cols, diag, anti = set(), set(), set()
    def place(r):
        nonlocal nodes
        nodes += 1
        if r == n:
            solutions.append(queens[:])
            return
        for c in range(n):
            if c in cols or (r - c) in diag or (r + c) in anti:
                continue                                                      # prune
            cols.add(c); diag.add(r - c); anti.add(r + c); queens.append(c)      # choose
            place(r + 1)                                                         # explore
            cols.remove(c); diag.remove(r - c); anti.remove(r + c); queens.pop() # un-choose
    place(0)
    return solutions, nodes

def queens_brute(n):
    return [list(p) for p in itertools.permutations(range(n))
            if len({r - c for r, c in enumerate(p)}) == n and len({r + c for r, c in enumerate(p)}) == n]

counts = [len(solve_n_queens(n)[0]) for n in range(1, 9)]
print("solutions for n = 1..8:", counts)
assert counts == [1, 0, 0, 2, 10, 4, 40, 92]
sols6, nodes6 = solve_n_queens(6)
assert sorted(sols6) == sorted(queens_brute(6))
print(f"n = 6: {len(sols6)} solutions; backtracking visited {nodes6} nodes vs {math.factorial(6)} brute-force boards")

In [ ]:
from matplotlib.colors import ListedColormap
fig, axes = plt.subplots(1, len(sols6), figsize=(11, 3))
for ax, sol in zip(axes, sols6):
    board = [[(r + c) % 2 for c in range(6)] for r in range(6)]
    ax.imshow(board, cmap=ListedColormap(["#f7f6f2", "#dcdad3"]))
    ax.add_patch(plt.Rectangle((-0.5, -0.5), 6, 6, fill=False, ec="#9a988f", lw=1))
    ax.scatter(sol, range(6), s=260, color=BLUE, zorder=3)
    ax.axis("off")
    ax.set_title("columns " + "".join(map(str, sol)), fontsize=10)
fig.suptitle("All 4 solutions of 6-Queens (row r holds a queen in column sol[r])")
plt.tight_layout(); plt.show()

**Complexity:** O(n!) time in the worst case (far less in practice thanks to pruning), O(n) space for the sets and the recursion.

### Problem 6 — Word search in a grid (LeetCode 79)

Can `word` be spelled by walking up/down/left/right through adjacent cells, never reusing a cell?

**Brute force:** list every simple path of length `len(word)` from every cell and compare. **Backtracking:** the same walk, but abandon a path at the first mismatched letter. Choose = overwrite the cell with `#` (marks it used, no extra `visited` set); un-choose = put the letter back. Two cheap pre-checks prune whole searches: the word cannot be longer than the grid, and the grid must contain enough copies of each letter.

In [ ]:
DIRS = ((1, 0), (-1, 0), (0, 1), (0, -1))

def word_search(board, word):
    rows, cols = len(board), len(board[0])
    if len(word) > rows * cols:
        return False, []
    have = Counter(ch for row in board for ch in row)
    if any(have[ch] < k for ch, k in Counter(word).items()):
        return False, []
    path = []
    def dfs(r, c, i):
        if not (0 <= r < rows and 0 <= c < cols) or board[r][c] != word[i]:
            return False
        if i == len(word) - 1:
            path.append((r, c))
            return True
        board[r][c] = "#"                                          # choose: mark used
        path.append((r, c))
        found = any(dfs(r + dr, c + dc, i + 1) for dr, dc in DIRS)   # explore
        board[r][c] = word[i]                                      # un-choose: restore
        if not found:
            path.pop()
        return found
    found = any(dfs(r, c, 0) for r in range(rows) for c in range(cols))
    return found, path

grid = [list("ABCE"), list("SFCS"), list("ADEE")]
snapshot = [row[:] for row in grid]
for w, expected in [("ABCCED", True), ("SEE", True), ("ABCB", False), ("ASADFBCCEESE", True)]:
    found, path = word_search(grid, w)
    print(f"{w!r:9} -> {found}   path: {path}")
    assert found == expected
assert grid == snapshot                              # every "#" was restored
assert word_search([["a"]], "a")[0] and not word_search([["a"]], "aa")[0]

In [ ]:
found, path = word_search(grid, "ABCCED")
fig, ax = plt.subplots(figsize=(5, 3.8))
for r, row in enumerate(grid):
    for c, ch in enumerate(row):
        on_path = (r, c) in path
        ax.add_patch(plt.Rectangle((c, -r), 0.94, 0.94, fc="#cde2fb" if on_path else "#f0efec", ec="white"))
        ax.text(c + 0.47, -r + 0.47, ch, ha="center", va="center", fontsize=14,
                fontweight="bold" if on_path else "normal")
        if on_path:
            ax.text(c + 0.08, -r + 0.84, str(path.index((r, c)) + 1), fontsize=8, color=MUTED, va="top")
for (r1, c1), (r2, c2) in zip(path, path[1:]):
    ax.annotate("", xy=(c2 + 0.47, -r2 + 0.47), xytext=(c1 + 0.47, -r1 + 0.47),
                arrowprops=dict(arrowstyle="->", color=ORANGE, lw=2, shrinkA=14, shrinkB=14))
ax.set_xlim(-0.1, 4); ax.set_ylim(-2.1, 1); ax.set_aspect("equal"); ax.axis("off")
ax.set_title("Word search: the path that spells ABCCED")
plt.show()

**Complexity:** O(R · C · 3^L) time — L letters, each step has at most 3 new directions (it never goes straight back) — and O(L) space for the recursion. Restoring the cell is not optional: skip it and the next starting cell sees a grid full of `#`.

## Try it yourself

**1. Permutations with duplicates (LeetCode 47).** `[1, 1, 2]` has only 3 distinct orderings. Hint: sort, and skip `nums[i]` when it equals `nums[i - 1]` and `nums[i - 1]` is not in use (that branch was already explored).

In [ ]:
# Solution — try it yourself first, then run this
def permute_unique(nums):
    nums, result, path = sorted(nums), [], []
    used = [False] * len(nums)
    def backtrack():
        if len(path) == len(nums):
            result.append(path[:])
            return
        for i in range(len(nums)):
            if used[i] or (i > 0 and nums[i] == nums[i - 1] and not used[i - 1]):
                continue
            used[i] = True; path.append(nums[i])
            backtrack()
            path.pop(); used[i] = False
    backtrack()
    return result

print(permute_unique([1, 1, 2]))
assert permute_unique([1, 1, 2]) == [[1, 1, 2], [1, 2, 1], [2, 1, 1]]
assert sorted(map(tuple, permute_unique([3, 1, 3, 1, 2]))) == sorted(set(itertools.permutations([3, 1, 3, 1, 2])))

**2. Letter combinations of a phone number (LeetCode 17).** `"23"` → `ad ae af bd be bf cd ce cf`. Empty input returns `[]`.

In [ ]:
# Solution — try it yourself first, then run this
PHONE = {"2": "abc", "3": "def", "4": "ghi", "5": "jkl", "6": "mno", "7": "pqrs", "8": "tuv", "9": "wxyz"}

def letter_combinations(digits):
    if not digits:
        return []
    result = []
    def backtrack(i, s):
        if i == len(digits):
            result.append(s)
            return
        for ch in PHONE[digits[i]]:
            backtrack(i + 1, s + ch)
    backtrack(0, "")
    return result

print(letter_combinations("23"))
assert letter_combinations("23") == ["ad", "ae", "af", "bd", "be", "bf", "cd", "ce", "cf"]
assert letter_combinations("") == [] and len(letter_combinations("79")) == 16

**3. Palindrome partitioning (LeetCode 131).** Split `"aab"` into pieces that are all palindromes: `[["a", "a", "b"], ["aa", "b"]]`. Hint: the choice at each step is "where does the next piece end?".

In [ ]:
# Solution — try it yourself first, then run this
def palindrome_partitions(s):
    result, path = [], []
    def backtrack(start):
        if start == len(s):
            result.append(path[:])
            return
        for end in range(start + 1, len(s) + 1):
            piece = s[start:end]
            if piece == piece[::-1]:                 # prune: only palindromic pieces
                path.append(piece); backtrack(end); path.pop()
    backtrack(0)
    return result

print(palindrome_partitions("aab"))
assert palindrome_partitions("aab") == [["a", "a", "b"], ["aa", "b"]]
assert palindrome_partitions("a") == [["a"]] and len(palindrome_partitions("aaa")) == 4

## Say it in an interview

**30-second answer — recursion:** "A recursive function has a base case that answers the smallest input directly and a recursive case that shrinks the input and combines the results. Each call adds a stack frame, so the space is O(depth). Python caps the depth at about 1000 and has no tail-call optimisation, so for deep inputs I use a loop or an explicit stack."

**30-second answer — backtracking:** "Backtracking is depth-first search over a tree of choices: choose, explore, un-choose, and prune any branch that can no longer lead to a valid answer. Time is the number of tree nodes times the work per node: 2^n subsets, n! permutations, Catalan numbers for parentheses. Space is the recursion depth plus the current path."

**Key numbers:** 2^n subsets · n! permutations · Catalan(n) ≈ 4^n / n^1.5 valid parenthesis strings · 8-Queens has 92 solutions · default recursion limit 1000.

**Follow-ups they will ask:**
- *"Why is your memoised version still slow / crashing?"* — memo removes repeated work, not depth; go bottom-up.
- *"How do you avoid duplicate combinations?"* — a `start` index (combinations), or sort + skip equal neighbours (inputs with duplicates).
- *"Where exactly do you prune?"* — before recursing: sorted candidates + `break`, conflict sets in N-Queens, counts in parentheses, letter counts in word search.

**Traps:** `result.append(path)` without a copy · forgetting to un-choose (restore the grid cell, pop the path, clear `used`) · `i + 1` vs `i` when reuse is allowed · mutable default arguments as a memo.

## Pattern cheat sheet

| Signal in the problem | Pattern | Key trick | Time |
|---|---|---|---|
| "all subsets / power set" | backtracking with `start` index | every node is an answer | O(n · 2^n) |
| "all orderings" | backtracking with `used[]` | skip used; with duplicates sort + skip equal neighbour | O(n · n!) |
| "combinations summing to target" | `start` index, recurse with `i` if reuse allowed | sort + `break` when too big | exponential |
| "all valid strings under a rule" | build left to right, never create an invalid prefix | counters (open/close) | ~ output size |
| "place items with conflicts" | one decision per row/slot + conflict sets | `r - c`, `r + c` diagonals | ~O(n!) |
| "path in a grid without reuse" | DFS + mark / restore cell | letter-count pre-check | O(RC · 3^L) |
| the same subproblem repeats | memoise → dynamic programming | `@lru_cache`, then bottom-up | states × work |

## Next

- [09 · Trees and BST](09_trees_and_bst.ipynb) — recursion on trees: every problem is "what do I need from my children?"
- [11 · Graphs](11_graphs_bfs_dfs_topo_dijkstra.ipynb) — DFS is this backtracking walk on a graph.
- [12 · Dynamic programming](12_dynamic_programming.ipynb) — memoisation grown up.
- [Functions, args, scope and lambda](../01_python_basics/06_functions_args_scope_lambda.ipynb) · [Decorators](../01_python_basics/12_decorators_and_context_managers.ipynb) (how `lru_cache` wraps a function)
- Theory: [DSA basics course — recursion chapter](../../dsa_basics/index.html)